# Flow

In [ ]:
✅ Tokenization
✅ Embeddings
✅ Positional Encoding
✅ Self-Attention
✅ Multi-Head Attention
✅ FFN
✅ LayerNorm
✅ Encoder Block
✅ Encoder
✅ Causal / Masked Self-Attention
✅ Decoder MHA
✅ Cross-Attention 
✅ Decoder Block

        ↓ NEXT

⬜ Full Transformer  ---> This NoteBook
⬜ Teacher Forcing
⬜ Output Projection + Softmax
⬜ Cross-Entropy Loss
⬜ Backpropagation
⬜ Tiny Transformer Training
⬜ Text Generation
⬜ GPT / Decoder-only Architecture
⬜ BPE Tokenization
⬜ RoPE
⬜ RMSNorm
⬜ SwiGLU
⬜ GQA / MQA
⬜ KV Cache
⬜ Tiny GPT
⬜ Fine-tuning
⬜ LoRA
⬜ RAG
⬜ Production LLM System

# Problem Statement

# Function

In [16]:
def token_embed(token_ids, embedding_matrix,d_model):
    return embedding_matrix[token_ids]*np.sqrt(d_model)

In [9]:
import numpy as np

def positional_encoding(seq_length: int, d_model: int) -> np.ndarray:
    """
    Returns the sinusoidal position matrix.
    """

    position = []

    for pos in range(seq_length):

        arr = []

        for i in range(d_model):

            if i % 2 == 0:
                value = np.sin(
                    pos / (10000 ** (i / d_model))
                )
            else:
                value = np.cos(
                    pos / (10000 ** ((i - 1) / d_model))
                )

            arr.append(value)

        position.append(arr)

    return np.array(position)

# Encoder block

In [28]:
import numpy as np

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """
    Numerically stable softmax.
    """
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def multi_head_attention(Q,K,V,W_q,W_k,W_v,W_o,num_heads):
    
    batch_size,seq_len,d_model=Q.shape

    assert d_model % num_heads == 0, \
        "d_model must be divisible by num_heads"

    d_head = d_model // num_heads

    #Linear Projection
    
    Q_new = Q @ W_q
    K_new = K @ W_k
    V_new = V @ W_v

    # split into multiple head
    # for eq x in shape of (1,3,4) --> (1,3,2,2)
    Q_heads=Q_new.reshape(batch_size,seq_len,num_heads,d_head)
    K_heads=K_new.reshape(batch_size,seq_len,num_heads,d_head)
    V_heads=V_new.reshape(batch_size,seq_len,num_heads,d_head)

    # Move heads before sequence dimension
    # Before:(batch, seq_length, num_heads, d_head)
    # After:(batch, num_heads, seq_length, d_head)
    Q_heads=Q_heads.transpose(0,2,1,3)
    K_heads=K_heads.transpose(0,2,1,3)
    V_heads=V_heads.transpose(0,2,1,3)


    # Scaled dot-product attention Q.K.T
    # K_heads.transpose(-2, -1):
    # (batch, heads, seq_length, d_head)
    #                    ↓
    # (batch, heads, d_head, seq_length)
    scores = Q_heads @ K_heads.transpose(0, 1, 3, 2)
    # Scale by sqrt(d_head)
    scores = scores / np.sqrt(d_head)

    # Softmax over keys
    attention_weights = softmax(scores, axis=-1)

    # Weighted sum of V

    head_outputs = attention_weights @ V_heads

    # Shape:
    # (batch, heads, seq_length, d_head)

    # Move dimensions back

    head_outputs = head_outputs.transpose(0, 2, 1, 3)

    # Shape:
    # (batch, seq_length, heads, d_head)

    # Concatenate all heads

    head_outputs = head_outputs.reshape(
        batch_size, seq_len, d_model
    )
    # Output projection
    output = head_outputs @ W_o

    return output

def residual(x,att_x):
    return x+att_x

def layer_norm(x, gamma, beta, eps=1e-6):
    mean=np.mean(x,axis=-1,keepdims=True)
    var=np.var(x,axis=-1,keepdims=True)
    x_norm=(x-mean)/np.sqrt(var+eps)
    output=(gamma*x_norm)+beta
    return output

def relu(x):
    return np.maximum(0, x)

def feed_forward(x,W1,b1,W2,b2):
    h1 = x @ W1
    h1 = h1 + b1
    h1 = relu(h1)
    out = h1 @ W2
    out = out + b2
    return out

def encoder_block(x: np.ndarray, W_q: np.ndarray, W_k: np.ndarray, W_v: np.ndarray,
                  W_o: np.ndarray, W1: np.ndarray, b1: np.ndarray,
                  W2: np.ndarray, b2: np.ndarray, gamma1: np.ndarray,
                  beta1: np.ndarray, gamma2: np.ndarray, beta2: np.ndarray,
                  num_heads: int) -> np.ndarray:
    """
    Returns the post-normalized Transformer encoder states.
    """
    # calculate dim_model.
    
    multi=multi_head_attention(x,x,x,W_q,W_k,W_v,W_o,num_heads)
    first_resi=residual(x,multi)
    fisrt_layer=layer_norm(first_resi,gamma1,beta1)
    # layer_norm(x, gamma, beta, eps=1e-6)
    feed_fn=feed_forward(fisrt_layer,W1,b1,W2,b2)
    second_resi=residual(fisrt_layer,feed_fn)
    second_layer=layer_norm(second_resi,gamma2,beta2)

    return second_layer
    

# Decoder block

In [34]:
# def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
#     """
#     Numerically stable softmax.
#     """
#     x = x - np.max(x, axis=axis, keepdims=True)
#     exp_x = np.exp(x)
#     softmax=exp_x / np.sum(exp_x, axis=axis, keepdims=True)
#     return softmax

def causal_mask(seq_len):
    causal_mask = np.triu(
        np.full((seq_len, seq_len), -np.inf),
        k=1
    )
    return causal_mask

def layer_norm(x: np.ndarray, gamma: np.ndarray, beta: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """
    Returns the last-axis normalized array.
    """
    mean=np.mean(x,axis=-1,keepdims=True)
    var=np.var(x,axis=-1,keepdims=True)
    x_norm=(x-mean)/np.sqrt(var+eps)
    layernorm=(gamma*x_norm)+beta
    return layernorm

# def relu(x):
#     return np.maximum(0, x)

def feed_forward(x: np.ndarray, W1: np.ndarray, b1: np.ndarray,
                 W2: np.ndarray, b2: np.ndarray) -> np.ndarray:
    """
    Returns the position-wise feed-forward output.
    """
    h1=np.add((x@W1),b1)
    h1_relu=relu(h1)
    ffn=np.add((h1_relu@W2),b2)
    return ffn
# def residual(x,att_x):
#     return x+att_x
    
def decoder_multi_head_attention(
    X,
    wq,
    wk,
    wv,
    wo,
    num_heads
):
    # Create QKV from input X
    Q=X@wq
    K=X@wk
    V=X@wv
    # define shape 
    batch_size, seq_length, d_model = Q.shape
    # define d_head
    d_head=d_model//num_heads
    # creation of heads
    Q_heads = Q.reshape(
        batch_size, seq_length, num_heads, d_head
    )

    K_heads = K.reshape(
        batch_size, seq_length, num_heads, d_head
    )

    V_heads = V.reshape(
        batch_size, seq_length, num_heads, d_head
    )
    # transpose
    Q_heads = Q_heads.transpose(0, 2, 1, 3)
    K_heads = K_heads.transpose(0, 2, 1, 3)
    V_heads = V_heads.transpose(0, 2, 1, 3)

    # q and kt dot product
    scores = Q_heads @ K_heads.transpose(0, 1, 3, 2)

    # Scale by sqrt(d_head)
    scores = scores / np.sqrt(d_head)

    # Causal mask
    causal = causal_mask(seq_length)

    # Apply mask
    masked = scores + causal

    #Soft masx
    attention_weights = softmax(masked, axis=-1)
    # weigthed sum of v
    head_outputs = attention_weights @ V_heads
    # transpose
    head_outputs = head_outputs.transpose(0, 2, 1, 3)
    #concate
    head_outputs = head_outputs.reshape(
        batch_size, seq_length, d_model
    )

    # Output projection

    output_self = head_outputs @ wo

    return output_self

def cross_attention(
    decoder_input,
    encoder_output,
    Wq,
    Wk,
    Wv,
    Wo,
    num_heads
):
    Q=decoder_input @ Wq
    K=encoder_output @ Wk
    V=encoder_output @ Wv

    batch_size, seq_length, d_model = Q.shape
    batch_size_en, seq_length_en, d_model_en = K.shape
    # define d_head
    d_head=d_model//num_heads

    Q_head=Q.reshape(batch_size,seq_length,num_heads,d_head)
    K_head=K.reshape(batch_size_en,seq_length_en,num_heads,d_head)
    V_head=V.reshape(batch_size_en,seq_length_en,num_heads,d_head)

    Q_head=Q_head.transpose(0,2,1,3)
    K_head=K_head.transpose(0,2,1,3)
    V_head=V_head.transpose(0,2,1,3)

    # q and kt dot product
    score=Q_head @ K_head.transpose(0,1,3,2)
    score=score/np.sqrt(d_head)

    # apply softmax
    score_soft=softmax(score,axis=-1)
    attentions=score_soft @ V_head

    head_output=attentions.transpose(0,2,1,3)
    head_output=head_output.reshape(batch_size, seq_length, d_model)
    
    output_cross=head_output@Wo
    return output_cross

def decoder_block(
    decoder_input,
    encoder_output,

    Wq_self,
    Wk_self,
    Wv_self,
    Wo_self,

    Wq_cross,
    Wk_cross,
    Wv_cross,
    Wo_cross,

    W1,
    b1,
    W2,
    b2,

    gamma1,
    beta1,
    gamma2,
    beta2,
    gamma3,
    beta3,

    num_heads
):
    self_attention=decoder_multi_head_attention(decoder_input,Wq_self,Wk_self,Wv_self,Wo_self,num_heads)
    first_residual = residual(decoder_input, self_attention)
    first_layer = layer_norm(first_residual, gamma1, beta1)
    
    cross_attn = cross_attention(
        first_layer,
        encoder_output,
        Wq_cross,
        Wk_cross,
        Wv_cross,
        Wo_cross,
        num_heads
    )
    
    second_residual = residual(first_layer, cross_attn)
    second_layer = layer_norm(second_residual, gamma2, beta2)
    
    ffn = feed_forward(second_layer, W1, b1, W2, b2)
    
    final_res = residual(second_layer, ffn)
    final_norm = layer_norm(final_res, gamma3, beta3)
    
    return final_norm

# Main

In [26]:
import numpy as np

np.random.seed(42)

# Dimensions
batch_size = 1

source_seq_len = 4
target_seq_len = 3

vocab_size = 10
d_model = 4
d_ff = 8
num_heads = 2

# Token IDs
source_tokens = np.array([
    [2, 5, 7, 3]
])

target_tokens = np.array([
    [1, 8, 4]
])

# Embedding matrix
embedding_matrix = np.random.randn(
    vocab_size,
    d_model
)

W_vocab = np.random.randn(
    d_model,
    vocab_size
)

In [ ]:
Wq_enc = np.random.randn(d_model, d_model)
Wk_enc = np.random.randn(d_model, d_model)
Wv_enc = np.random.randn(d_model, d_model)
Wo_enc = np.random.randn(d_model, d_model)

W1_enc = np.random.randn(d_model, d_ff)
b1_enc = np.random.randn(d_ff)

W2_enc = np.random.randn(d_ff, d_model)
b2_enc = np.random.randn(d_model)

gamma1_enc = np.ones(d_model)
beta1_enc = np.zeros(d_model)

gamma2_enc = np.ones(d_model)
beta2_enc = np.zeros(d_model)

Wq_self = np.random.randn(d_model, d_model)
Wk_self = np.random.randn(d_model, d_model)
Wv_self = np.random.randn(d_model, d_model)
Wo_self = np.random.randn(d_model, d_model)

Wq_cross = np.random.randn(d_model, d_model)
Wk_cross = np.random.randn(d_model, d_model)
Wv_cross = np.random.randn(d_model, d_model)
Wo_cross = np.random.randn(d_model, d_model)

W1_dec = np.random.randn(d_model, d_ff)
b1_dec = np.random.randn(d_ff)

W2_dec = np.random.randn(d_ff, d_model)
b2_dec = np.random.randn(d_model)

gamma1_dec = np.ones(d_model)
beta1_dec = np.zeros(d_model)

gamma2_dec = np.ones(d_model)
beta2_dec = np.zeros(d_model)

gamma3_dec = np.ones(d_model)
beta3_dec = np.zeros(d_model)

In [36]:
# Token Embedding
token_embedding_source=token_embed(source_tokens, embedding_matrix,d_model)
token_embedding_target=token_embed(target_tokens, embedding_matrix,d_model)

# Positional encoding
positional_encoding_matrix = positional_encoding(
    max(source_seq_len, target_seq_len),
    d_model
)
# So we can use the same positional encoding matrix:
source_pe = positional_encoding_matrix[:source_seq_len]

target_pe = positional_encoding_matrix[:target_seq_len]

# target embedding + Position embedding

embedding=token_embedding_source+source_pe

decoder_input=token_embedding_target+target_pe

In [37]:
encoder_output=encoder_block(embedding,Wq_enc, Wk_enc, Wv_enc,Wo_enc, W1_enc, b1_enc,W2_enc, b2_enc,
                             gamma1_enc,beta1_enc, gamma2_enc, beta2_enc,num_heads)

In [41]:
decoder_output = decoder_block(decoder_input,encoder_output,Wq_self,Wk_self,Wv_self,Wo_self,Wq_cross,
    Wk_cross,Wv_cross,Wo_cross,W1_dec,b1_dec,W2_dec,b2_dec,
    gamma1_dec,beta1_dec,gamma2_dec,beta2_dec,gamma3_dec,beta3_dec,num_heads
)


In [ ]:
logits = decoder_output @ W_vocab

probabilities = softmax(logits, axis=-1)

print(probabilities.shape)
print(probabilities)
print(np.sum(probabilities, axis=-1))

In [66]:
probabilities

array([[[1.39398748e-03, 2.41938239e-04, 7.47565029e-04, 1.11863726e-03,
         7.06605382e-01, 1.15221612e-02, 9.18225233e-04, 1.29625341e-03,
         3.24454621e-03, 2.72911304e-01],
        [1.99328014e-03, 5.87830548e-04, 1.22258424e-03, 3.65692414e-03,
         5.94890315e-01, 2.51727053e-02, 1.54573683e-03, 1.10735133e-03,
         4.32498319e-03, 3.65498290e-01],
        [2.76984337e-03, 1.12187264e-03, 1.84771035e-03, 8.03513731e-03,
         5.13321073e-01, 4.02813244e-02, 2.31473712e-03, 1.13789929e-03,
         5.57436042e-03, 4.23596042e-01]]])

In [70]:
probabilities[0,1,2]

IndexError: index 8 is out of bounds for axis 1 with size 3

# Cross Entropy

# Complete Mental Flow
Your function should therefore think like this:

In [102]:
def cross_entropy_loss(probabilities, target_labels):
    batch_size, seq_len, vocab_size = probabilities.shape

    correct_probs = []

    for batch in range(batch_size):
        for pos in range(seq_len):
            target_token = target_labels[batch, pos]

            prob = probabilities[batch, pos, target_token]

            correct_probs.append(prob)

    correct_probs = np.array(correct_probs)

    # Numerical stability
    eps = 1e-12
    correct_probs = np.clip(correct_probs, eps, 1.0)

    losses = -np.log(correct_probs)

    return np.mean(losses)

In [ ]:
# without for lopp numpy version

def cross_entropy_loss(probabilities, target_labels):
    batch_indices = np.arange(probabilities.shape[0])[:, None]
    seq_indices = np.arange(probabilities.shape[1])[None, :]

    correct_probs = probabilities[
        batch_indices,
        seq_indices,
        target_labels
    ]

    eps = 1e-12
    correct_probs = np.clip(correct_probs, eps, 1.0)

    return -np.mean(np.log(correct_probs))

In [103]:
cross_entropy_loss(probabilities, target_tokens)

np.float64(4.812342951819254)

In [ ]:
import numpy as np
probabilities = np.array([
    [
        [0.1, 0.7, 0.2],
        [0.6, 0.1, 0.3],
        [0.2, 0.2, 0.6]
    ],
    [
        [0.8, 0.1, 0.1],
        [0.2, 0.5, 0.3],
        [0.1, 0.3, 0.6]
    ]
])

target_labels = np.array([
    [1, 0, 2],
    [0, 1, 2]
])

loss = cross_entropy_loss(
    probabilities,
    target_labels
)

print("Loss:", loss)

# Backpropagation

In [2]:
import numpy as np

def softmax_cross_entropy_gradient(probabilities, target_labels):
    batch_size, seq_len, vocab_size = probabilities.shape

    # Step 1: Create a one-hot array with the same shape as probabilities
    target_one_hot = np.zeros_like(probabilities)

    # Step 2: Create batch and sequence indices
    batch_indices = np.arange(batch_size)[:, None]
    seq_indices = np.arange(seq_len)[None, :]

    # Step 3: Set the correct token positions to 1
    target_one_hot[batch_indices, seq_indices, target_labels] = 1

    # Step 4: Calculate the gradient
    gradient_logits = (
        probabilities - target_one_hot
    ) / (batch_size * seq_len)

    return gradient_logits

In [1]:
import numpy as np
probabilities = np.array([
    [
        [0.1, 0.7, 0.2],
        [0.6, 0.1, 0.3],
        [0.2, 0.2, 0.6]
    ],
    [
        [0.8, 0.1, 0.1],
        [0.2, 0.5, 0.3],
        [0.1, 0.3, 0.6]
    ]
])

target_labels = np.array([
    [1, 0, 2],
    [0, 1, 2]
])

In [3]:
softmax_cross_entropy_gradient(probabilities, target_labels)

array([[[ 0.01666667, -0.05      ,  0.03333333],
        [-0.06666667,  0.01666667,  0.05      ],
        [ 0.03333333,  0.03333333, -0.06666667]],

       [[-0.03333333,  0.01666667,  0.01666667],
        [ 0.03333333, -0.08333333,  0.05      ],
        [ 0.01666667,  0.05      , -0.06666667]]])

# Backpropagation

In [4]:

import numpy as np

np.random.seed(42)


# --------------------------------------------------
# 1. SOFTMAX
# --------------------------------------------------
def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


# --------------------------------------------------
# 2. CROSS-ENTROPY LOSS
# --------------------------------------------------
def cross_entropy_loss(probabilities, target_labels):
    batch_size, seq_len, vocab_size = probabilities.shape

    batch_indices = np.arange(batch_size)[:, None]
    seq_indices = np.arange(seq_len)[None, :]

    correct_probs = probabilities[
        batch_indices,
        seq_indices,
        target_labels
    ]

    eps = 1e-12
    correct_probs = np.clip(correct_probs, eps, 1.0)

    return -np.mean(np.log(correct_probs))


# --------------------------------------------------
# 3. GRADIENT OF LOSS W.R.T. LOGITS
# --------------------------------------------------
def softmax_cross_entropy_gradient(probabilities, target_labels):
    batch_size, seq_len, vocab_size = probabilities.shape

    target_one_hot = np.zeros_like(probabilities)

    batch_indices = np.arange(batch_size)[:, None]
    seq_indices = np.arange(seq_len)[None, :]

    target_one_hot[
        batch_indices, seq_indices, target_labels
    ] = 1

    gradient_logits = (
        probabilities - target_one_hot
    ) / (batch_size * seq_len)

    return gradient_logits


# --------------------------------------------------
# 4. GRADIENTS OF OUTPUT PROJECTION
# --------------------------------------------------
def output_projection_backward(
    decoder_output,
    W_vocab,
    gradient_logits
):
    # Gradient w.r.t. vocabulary weights
    dW_vocab = np.einsum(
        "bsd,bsv->dv",
        decoder_output,
        gradient_logits
    )

    # Gradient w.r.t. vocabulary bias
    db_vocab = np.sum(
        gradient_logits,
        axis=(0, 1)
    )

    # Gradient flowing back into decoder output
    d_decoder_output = gradient_logits @ W_vocab.T

    return dW_vocab, db_vocab, d_decoder_output


# --------------------------------------------------
# 5. CREATE A SMALL TOY EXAMPLE
# --------------------------------------------------
batch_size = 2
seq_len = 3
d_model = 4
vocab_size = 6

# Pretend these came from your Transformer decoder
decoder_output = np.random.randn(
    batch_size, seq_len, d_model
)

# Trainable output projection parameters
W_vocab = np.random.randn(
    d_model, vocab_size
) * 0.1

b_vocab = np.zeros(vocab_size)

# Correct next-token IDs for each position
target_labels = np.array([
    [1, 2, 3],
    [2, 4, 5]
])


# --------------------------------------------------
# 6. TRAIN THE OUTPUT PROJECTION
# --------------------------------------------------
learning_rate = 0.1
epochs = 100

for epoch in range(epochs):

    # FORWARD PASS
    logits = decoder_output @ W_vocab + b_vocab

    probabilities = softmax(logits, axis=-1)

    loss = cross_entropy_loss(
        probabilities, target_labels
    )

    # BACKWARD PASS
    gradient_logits = softmax_cross_entropy_gradient(
        probabilities, target_labels
    )

    dW_vocab, db_vocab, d_decoder_output = (
        output_projection_backward(
            decoder_output,
            W_vocab,
            gradient_logits
        )
    )

    # UPDATE PARAMETERS
    W_vocab -= learning_rate * dW_vocab
    b_vocab -= learning_rate * db_vocab

    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss:.4f}")


# --------------------------------------------------
# 7. CHECK FINAL LOSS
# --------------------------------------------------
final_logits = decoder_output @ W_vocab + b_vocab
final_probabilities = softmax(final_logits, axis=-1)

final_loss = cross_entropy_loss(
    final_probabilities, target_labels
)

print("\nFinal loss:", round(final_loss, 4))
print("W_vocab shape:", W_vocab.shape)
print("b_vocab shape:", b_vocab.shape)
print("Gradient logits shape:", gradient_logits.shape)
print("Decoder output gradient shape:", d_decoder_output.shape)


Epoch   0 | Loss: 1.8718
Epoch  20 | Loss: 1.1391
Epoch  40 | Loss: 0.8113
Epoch  60 | Loss: 0.6391
Epoch  80 | Loss: 0.5325

Final loss: 0.4589
W_vocab shape: (4, 6)
b_vocab shape: (6,)
Gradient logits shape: (2, 3, 6)
Decoder output gradient shape: (2, 3, 4)


# Full code

In [9]:
import numpy as np

# ============================================================
# COMPLETE SMALL ENCODER-DECODER TRANSFORMER
# NumPy only; manual backpropagation for every trainable parameter
# Post-LayerNorm, learned token embeddings, sinusoidal positions
# ============================================================

class TinyTransformer:
    def __init__(
        self,
        vocab_size,
        d_model=8,
        num_heads=2,
        d_ff=16,
        seed=42,
        eps=1e-5
    ):
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.d_ff = d_ff
        self.eps = eps
        self.rng = np.random.default_rng(seed)
        self.params = {}
        self._initialize_parameters()

    def _weight(self, shape, fan_in):
        return self.rng.normal(0, np.sqrt(2.0 / fan_in), size=shape)

    def _linear_init(self, prefix, in_dim, out_dim):
        self.params[prefix + ".W"] = self._weight((in_dim, out_dim), in_dim)
        self.params[prefix + ".b"] = np.zeros(out_dim)

    def _norm_init(self, prefix):
        self.params[prefix + ".gamma"] = np.ones(self.d_model)
        self.params[prefix + ".beta"] = np.zeros(self.d_model)

    def _initialize_parameters(self):
        D, V, F = self.d_model, self.vocab_size, self.d_ff

        self.params["src_embedding"] = self._weight((V, D), D)
        self.params["tgt_embedding"] = self._weight((V, D), D)

        for prefix in ("enc.self", "dec.self", "dec.cross"):
            for projection in ("Wq", "Wk", "Wv", "Wo"):
                self._linear_init(f"{prefix}.{projection}", D, D)

        self._linear_init("enc.ffn1", D, F)
        self._linear_init("enc.ffn2", F, D)
        self._norm_init("enc.norm1")
        self._norm_init("enc.norm2")

        self._linear_init("dec.ffn1", D, F)
        self._linear_init("dec.ffn2", F, D)
        for i in (1, 2, 3):
            self._norm_init(f"dec.norm{i}")

        self._linear_init("vocab", D, V)

    # -------------------- basic forward ops --------------------

    def _positional_encoding(self, seq_len):
        D = self.d_model
        positions = np.arange(seq_len)[:, None]
        dims = np.arange(D)[None, :]
        rates = 1.0 / np.power(10000.0, (2 * (dims // 2)) / D)
        angles = positions * rates
        pe = np.zeros((seq_len, D))
        pe[:, 0::2] = np.sin(angles[:, 0::2])
        pe[:, 1::2] = np.cos(angles[:, 1::2])
        return pe

    def _linear(self, x, prefix):
        W, b = self.params[prefix + ".W"], self.params[prefix + ".b"]
        return x @ W + b, (x, W)

    def _linear_backward(self, dout, cache, prefix, grads):
        x, W = cache
        x_flat = x.reshape(-1, x.shape[-1])
        dout_flat = dout.reshape(-1, dout.shape[-1])
        grads[prefix + ".W"] = x_flat.T @ dout_flat
        grads[prefix + ".b"] = dout_flat.sum(axis=0)
        return dout @ W.T

    def _layer_norm(self, x, prefix):
        gamma = self.params[prefix + ".gamma"]
        beta = self.params[prefix + ".beta"]
        mean = x.mean(axis=-1, keepdims=True)
        centered = x - mean
        var = np.mean(centered * centered, axis=-1, keepdims=True)
        inv_std = 1.0 / np.sqrt(var + self.eps)
        xhat = centered * inv_std
        out = gamma * xhat + beta
        cache = (xhat, inv_std, gamma)
        return out, cache

    def _layer_norm_backward(self, dout, cache, prefix, grads):
        xhat, inv_std, gamma = cache
        axis = -1
        grads[prefix + ".gamma"] = np.sum(dout * xhat, axis=(0, 1))
        grads[prefix + ".beta"] = np.sum(dout, axis=(0, 1))

        dxhat = dout * gamma
        D = dxhat.shape[-1]
        sum_dxhat = np.sum(dxhat, axis=axis, keepdims=True)
        sum_dxhat_xhat = np.sum(dxhat * xhat, axis=axis, keepdims=True)
        dx = (inv_std / D) * (
            D * dxhat - sum_dxhat - xhat * sum_dxhat_xhat
        )
        return dx

    def _attention_forward(self, xq, xkv, prefix, causal=False):
        # xq: (B,T,D); xkv: (B,S,D)
        B, T, D = xq.shape
        S = xkv.shape[1]
        H, Dh = self.num_heads, self.d_head

        q, qcache = self._linear(xq, prefix + ".Wq")
        k, kcache = self._linear(xkv, prefix + ".Wk")
        v, vcache = self._linear(xkv, prefix + ".Wv")

        q = q.reshape(B, T, H, Dh).transpose(0, 2, 1, 3)
        k = k.reshape(B, S, H, Dh).transpose(0, 2, 1, 3)
        v = v.reshape(B, S, H, Dh).transpose(0, 2, 1, 3)

        scores = q @ k.transpose(0, 1, 3, 2) / np.sqrt(Dh)
        if causal:
            mask = np.triu(np.ones((T, S), dtype=bool), k=1)
            scores = np.where(mask[None, None, :, :], -1e9, scores)

        weights = self._softmax(scores, axis=-1)
        context = weights @ v
        merged = context.transpose(0, 2, 1, 3).reshape(B, T, D)
        out, ocache = self._linear(merged, prefix + ".Wo")

        cache = {
            "qcache": qcache, "kcache": kcache, "vcache": vcache,
            "ocache": ocache, "q": q, "k": k, "v": v,
            "weights": weights, "context": context, "merged": merged,
            "xq_shape": xq.shape, "xkv_shape": xkv.shape,
            "prefix": prefix, "causal": causal
        }
        return out, cache

    @staticmethod
    def _softmax(x, axis=-1):
        x = x - np.max(x, axis=axis, keepdims=True)
        ex = np.exp(x)
        return ex / ex.sum(axis=axis, keepdims=True)

    def _attention_backward(self, dout, cache, grads):
        prefix = cache["prefix"]
        B, H, T, Dh = cache["q"].shape
        S = cache["k"].shape[2]
        D = self.d_model

        dmerged = self._linear_backward(dout, cache["ocache"], prefix + ".Wo", grads)
        dcontext = dmerged.reshape(B, T, H, Dh).transpose(0, 2, 1, 3)

        weights = cache["weights"]
        v = cache["v"]
        dweights = dcontext @ v.transpose(0, 1, 3, 2)
        dv = weights.transpose(0, 1, 3, 2) @ dcontext

        # Backprop through softmax: dx = p * (g - sum(g*p))
        dscores = weights * (
            dweights - np.sum(dweights * weights, axis=-1, keepdims=True)
        )
        if cache["causal"]:
            mask = np.triu(np.ones((T, S), dtype=bool), k=1)
            dscores = np.where(mask[None, None, :, :], 0.0, dscores)

        scale = 1.0 / np.sqrt(Dh)
        dq = (dscores @ cache["k"]) * scale
        dk = (dscores.transpose(0, 1, 3, 2) @ cache["q"]) * scale

        dq = dq.transpose(0, 2, 1, 3).reshape(B, T, D)
        dk = dk.transpose(0, 2, 1, 3).reshape(B, S, D)
        dv = dv.transpose(0, 2, 1, 3).reshape(B, S, D)

        dxq = self._linear_backward(dq, cache["qcache"], prefix + ".Wq", grads)
        dxk = self._linear_backward(dk, cache["kcache"], prefix + ".Wk", grads)
        dxv = self._linear_backward(dv, cache["vcache"], prefix + ".Wv", grads)
        return dxq, dxk + dxv

    def _ffn_forward(self, x, prefix):
        hpre, c1 = self._linear(x, prefix + ".ffn1")
        h = np.maximum(0.0, hpre)
        out, c2 = self._linear(h, prefix + ".ffn2")
        return out, (hpre, c1, c2)

    def _ffn_backward(self, dout, cache, prefix, grads):
        hpre, c1, c2 = cache
        dh = self._linear_backward(dout, c2, prefix + ".ffn2", grads)
        dhpre = dh * (hpre > 0.0)
        return self._linear_backward(dhpre, c1, prefix + ".ffn1", grads)

    # -------------------- complete forward pass --------------------

    def forward(self, source_tokens, target_input_tokens):
        source_tokens = np.asarray(source_tokens, dtype=int)
        target_input_tokens = np.asarray(target_input_tokens, dtype=int)
        if source_tokens.ndim != 2 or target_input_tokens.ndim != 2:
            raise ValueError("Token arrays must have shape (batch, sequence)")
        if source_tokens.shape[0] != target_input_tokens.shape[0]:
            raise ValueError("Source and target batch sizes must match")

        B, src_len = source_tokens.shape
        _, tgt_len = target_input_tokens.shape
        P = self.params

        src_pe = self._positional_encoding(src_len)[None, :, :]
        tgt_pe = self._positional_encoding(tgt_len)[None, :, :]

        src_x = P["src_embedding"][source_tokens] * np.sqrt(self.d_model) + src_pe
        tgt_x = P["tgt_embedding"][target_input_tokens] * np.sqrt(self.d_model) + tgt_pe

        # Encoder block, post-LN
        enc_attn, enc_attn_cache = self._attention_forward(
            src_x, src_x, "enc.self", causal=False
        )
        enc_res1 = src_x + enc_attn
        enc_n1, enc_n1_cache = self._layer_norm(enc_res1, "enc.norm1")

        enc_ffn, enc_ffn_cache = self._ffn_forward(enc_n1, "enc")
        enc_res2 = enc_n1 + enc_ffn
        encoder_output, enc_n2_cache = self._layer_norm(enc_res2, "enc.norm2")

        # Decoder masked self-attention
        dec_attn, dec_attn_cache = self._attention_forward(
            tgt_x, tgt_x, "dec.self", causal=True
        )
        dec_res1 = tgt_x + dec_attn
        dec_n1, dec_n1_cache = self._layer_norm(dec_res1, "dec.norm1")

        # Decoder cross-attention
        cross, cross_cache = self._attention_forward(
            dec_n1, encoder_output, "dec.cross", causal=False
        )
        dec_res2 = dec_n1 + cross
        dec_n2, dec_n2_cache = self._layer_norm(dec_res2, "dec.norm2")

        # Decoder FFN
        dec_ffn, dec_ffn_cache = self._ffn_forward(dec_n2, "dec")
        dec_res3 = dec_n2 + dec_ffn
        decoder_output, dec_n3_cache = self._layer_norm(dec_res3, "dec.norm3")

        logits, vocab_cache = self._linear(decoder_output, "vocab")
        probabilities = self._softmax(logits, axis=-1)

        cache = {
            "source_tokens": source_tokens,
            "target_input_tokens": target_input_tokens,
            "src_x_shape": src_x.shape,
            "tgt_x_shape": tgt_x.shape,
            "enc_attn_cache": enc_attn_cache,
            "enc_n1_cache": enc_n1_cache,
            "enc_ffn_cache": enc_ffn_cache,
            "enc_n2_cache": enc_n2_cache,
            "dec_attn_cache": dec_attn_cache,
            "dec_n1_cache": dec_n1_cache,
            "cross_cache": cross_cache,
            "dec_n2_cache": dec_n2_cache,
            "dec_ffn_cache": dec_ffn_cache,
            "dec_n3_cache": dec_n3_cache,
            "vocab_cache": vocab_cache,
            "src_x": src_x,
            "tgt_x": tgt_x,
            "enc_res1": enc_res1,
            "enc_n1": enc_n1,
            "enc_res2": enc_res2,
            "encoder_output": encoder_output,
            "dec_res1": dec_res1,
            "dec_n1": dec_n1,
            "dec_res2": dec_res2,
            "dec_n2": dec_n2,
            "dec_res3": dec_res3,
            "decoder_output": decoder_output,
            "logits": logits,
            "probabilities": probabilities
        }
        return probabilities, cache

    # -------------------- loss and complete backward pass --------------------

    @staticmethod
    def loss(probabilities, target_labels):
        B, S, V = probabilities.shape
        target_labels = np.asarray(target_labels, dtype=int)
        bi = np.arange(B)[:, None]
        si = np.arange(S)[None, :]
        correct = probabilities[bi, si, target_labels]
        return -np.mean(np.log(np.clip(correct, 1e-12, 1.0)))

    def backward(self, target_labels, cache):
        P = self.params
        grads = {}
        probs = cache["probabilities"]
        target_labels = np.asarray(target_labels, dtype=int)
        B, S, V = probs.shape

        # dL/dlogits for mean cross-entropy
        one_hot = np.zeros_like(probs)
        one_hot[np.arange(B)[:, None], np.arange(S)[None, :], target_labels] = 1
        dlogits = (probs - one_hot) / (B * S)

        # Output projection
        d_decoder = self._linear_backward(
            dlogits, cache["vocab_cache"], "vocab", grads
        )

        # Decoder final LayerNorm and FFN residual
        d_dec_res3 = self._layer_norm_backward(
            d_decoder, cache["dec_n3_cache"], "dec.norm3", grads
        )
        d_dec_n2 = d_dec_res3.copy()
        d_dec_n2 += self._ffn_backward(
            d_dec_res3, cache["dec_ffn_cache"], "dec", grads
        )

        # Decoder second LayerNorm and cross-attention residual
        d_dec_res2 = self._layer_norm_backward(
            d_dec_n2, cache["dec_n2_cache"], "dec.norm2", grads
        )
        d_dec_n1 = d_dec_res2.copy()
        d_cross_q, d_encoder_from_cross = self._attention_backward(
            d_dec_res2, cache["cross_cache"], grads
        )
        d_dec_n1 += d_cross_q

        # Decoder first LayerNorm and masked self-attention residual
        d_dec_res1 = self._layer_norm_backward(
            d_dec_n1, cache["dec_n1_cache"], "dec.norm1", grads
        )
        d_tgt_x = d_dec_res1.copy()
        d_self_q, d_self_kv = self._attention_backward(
            d_dec_res1, cache["dec_attn_cache"], grads
        )
        d_tgt_x += d_self_q + d_self_kv

        # Encoder backward: include gradient arriving from cross-attention
        d_enc_output = d_encoder_from_cross
        d_enc_res2 = self._layer_norm_backward(
            d_enc_output, cache["enc_n2_cache"], "enc.norm2", grads
        )
        d_enc_n1 = d_enc_res2.copy()
        d_enc_n1 += self._ffn_backward(
            d_enc_res2, cache["enc_ffn_cache"], "enc", grads
        )

        d_enc_res1 = self._layer_norm_backward(
            d_enc_n1, cache["enc_n1_cache"], "enc.norm1", grads
        )
        d_src_x = d_enc_res1.copy()
        d_enc_self_q, d_enc_self_kv = self._attention_backward(
            d_enc_res1, cache["enc_attn_cache"], grads
        )
        d_src_x += d_enc_self_q + d_enc_self_kv

        # Accumulate gradients for repeated token IDs
        grads["src_embedding"] = np.zeros_like(P["src_embedding"])
        np.add.at(
            grads["src_embedding"],
            cache["source_tokens"],
            d_src_x * np.sqrt(self.d_model)
        )

        grads["tgt_embedding"] = np.zeros_like(P["tgt_embedding"])
        np.add.at(
            grads["tgt_embedding"],
            cache["target_input_tokens"],
            d_tgt_x * np.sqrt(self.d_model)
        )

        return grads

    def train_step(self, source_tokens, target_input_tokens, target_labels, lr=0.01):
        probabilities, cache = self.forward(source_tokens, target_input_tokens)
        loss = self.loss(probabilities, target_labels)
        grads = self.backward(target_labels, cache)

        # Basic global norm clipping to avoid unstable large updates
        total_sq_norm = sum(np.sum(g * g) for g in grads.values())
        grad_norm = np.sqrt(total_sq_norm)
        clip_scale = min(1.0, 1.0 / (grad_norm + 1e-12))

        for name in self.params:
            self.params[name] -= lr * grads[name] * clip_scale

        return loss, grad_norm

    def predict(self, source_tokens, target_input_tokens):
        probabilities, _ = self.forward(source_tokens, target_input_tokens)
        return np.argmax(probabilities, axis=-1), probabilities


# ============================================================
# NUMERICAL GRADIENT CHECK (central finite differences)
# ============================================================

def numerical_gradient(model, source_tokens, target_input_tokens,
                       target_labels, param_name, index, eps=1e-5):
    param = model.params[param_name]
    original = param[index]

    param[index] = original + eps
    p_plus, _ = model.forward(source_tokens, target_input_tokens)
    loss_plus = model.loss(p_plus, target_labels)

    param[index] = original - eps
    p_minus, _ = model.forward(source_tokens, target_input_tokens)
    loss_minus = model.loss(p_minus, target_labels)

    param[index] = original
    return (loss_plus - loss_minus) / (2 * eps)


# ============================================================
# TOY NEXT-TOKEN TASK
# target sequence: [BOS, a, b] -> labels [a, b, EOS]
# IDs: 0=PAD, 1=BOS, 2=a, 3=b, 4=EOS, 5=c
# ============================================================

if __name__ == "__main__":
    model = TinyTransformer(
        vocab_size=6,
        d_model=8,
        num_heads=2,
        d_ff=16,
        seed=7
    )

    source_tokens = np.array([
        [2, 3, 4],
        [3, 2, 4]
    ])

    # Teacher forcing: decoder input starts with BOS
    target_input_tokens = np.array([
        [1, 2, 3],
        [1, 3, 2]
    ])

    # Correct next token at each target position
    target_labels = np.array([
        [2, 3, 4],
        [3, 2, 4]
    ])

    # Check representative gradients before training
    probabilities, cache = model.forward(source_tokens, target_input_tokens)
    analytic_grads = model.backward(target_labels, cache)

    checks = [
        ("vocab.W", (0, 0)),
        ("dec.ffn1.W", (0, 0)),
        ("dec.self.Wq.W", (0, 0)),
        ("enc.self.Wk.W", (0, 0)),
        ("src_embedding", (2, 0)),
        ("tgt_embedding", (1, 0)),
    ]

    print("Gradient checks (analytical vs numerical):")
    for name, idx in checks:
        numerical = numerical_gradient(
            model, source_tokens, target_input_tokens,
            target_labels, name, idx
        )
        analytical = analytic_grads[name][idx]
        rel_error = abs(analytical - numerical) / max(
            1e-8, abs(analytical) + abs(numerical)
        )
        print(
            f"{name}{idx}: analytical={analytical:.6e}, "
            f"numerical={numerical:.6e}, rel_error={rel_error:.3e}"
        )

    # Train the full model on the tiny synthetic task
    print("\nTraining:")
    for step in range(51):
        loss, grad_norm = model.train_step(
            source_tokens,
            target_input_tokens,
            target_labels,
            lr=0.03
        )
        if step % 50 == 0:
            print(f"step={step:3d}, loss={loss:.4f}, grad_norm={grad_norm:.4f}")

    predictions, final_probs = model.predict(source_tokens, target_input_tokens)
    print("\nTarget labels:\n", target_labels)
    print("Predicted token IDs:\n", predictions)
    print("Final loss:", model.loss(final_probs, target_labels))

Gradient checks (analytical vs numerical):
vocab.W(0, 0): analytical=-2.865666e-02, numerical=-2.865666e-02, rel_error=3.627e-10
dec.ffn1.W(0, 0): analytical=-1.164913e-02, numerical=-1.164913e-02, rel_error=1.874e-09
dec.self.Wq.W(0, 0): analytical=-9.633381e-05, numerical=-9.633383e-05, rel_error=1.089e-07
enc.self.Wk.W(0, 0): analytical=-1.055560e-02, numerical=-1.055560e-02, rel_error=3.164e-09
src_embedding(2, 0): analytical=1.287873e-01, numerical=1.287873e-01, rel_error=1.695e-10
tgt_embedding(1, 0): analytical=4.318667e-01, numerical=4.318667e-01, rel_error=2.650e-11

Training:
step=  0, loss=5.1330, grad_norm=8.6438
step= 50, loss=0.4475, grad_norm=1.1930

Target labels:
 [[2 3 4]
 [3 2 4]]
Predicted token IDs:
 [[2 3 4]
 [3 4 4]]
Final loss: 0.4330137668927609
